# Set Operations

## Quick notes
### What this covers
- `UNION`, `UNION ALL`, `INTERSECT`, `EXCEPT` and how they treat duplicates and NULLs.
- ORDER BY / LIMIT on a compound select, evaluation order, grouping with subqueries.
- Set operations vs joins, "what differs between two tables", emulating `FULL OUTER JOIN` and `INTERSECT ALL`.
### Key points
- Set operations stack result sets vertically (joins widen them horizontally).
- Both sides need the same number of columns. Columns are matched by position, names come from the first SELECT.
- `UNION`, `INTERSECT`, `EXCEPT` remove duplicates. `UNION ALL` keeps everything and is faster.
- ORDER BY / LIMIT go once, at the very end, and apply to the whole result.
### Gotchas (the part worth revising twice)
- Use `UNION ALL` unless you really want de-duplication; plain `UNION` has to sort or hash to remove duplicates.
- Columns match by position, not name. Swapping column order silently gives wrong data.
- SQLite has no `INTERSECT ALL` or `EXCEPT ALL`. Emulate them.
- In set operations two NULLs count as equal (unlike `=`).
- Compound selects run left to right with equal precedence in SQLite. Use subqueries to group.
- `EXCEPT` is not symmetric: `A EXCEPT B` differs from `B EXCEPT A`.
### When it matters
- Merging monthly tables, finding customers who did X but not Y, diffing two copies of a table, reconciliation.
### One-line recap example
```sql
SELECT email FROM newsletter EXCEPT SELECT email FROM unsubscribed;
```

## 01. Seed data
- Two lists of people with deliberate overlap, a duplicate, and one NULL.

In [1]:
# setup: an in-memory SQLite database and a %%sql cell magic that prints result tables
import sqlite3
from IPython.core.magic import register_cell_magic

con = sqlite3.connect(":memory:", isolation_level=None)   # autocommit, so BEGIN/COMMIT work as written
con.execute("PRAGMA foreign_keys = ON")

def _table(cols, rows):
    cells = [[("NULL" if v is None else str(v)) for v in r] for r in rows]
    widths = [max([len(c)] + [len(r[i]) for r in cells]) for i, c in enumerate(cols)]
    line = lambda vals: " | ".join(v.ljust(w) for v, w in zip(vals, widths))
    print(line(cols)); print("-+-".join("-" * w for w in widths))
    for r in cells: print(line(r))
    print(f"({len(rows)} row{'s' if len(rows) != 1 else ''})\n")

def _statements(text):
    """Split a cell into statements (handles several per line, triggers, quoted semicolons)."""
    out, start = [], 0
    for i, ch in enumerate(text):
        if ch == ";" and sqlite3.complete_statement(text[start:i + 1]):
            out.append(text[start:i + 1].strip()); start = i + 1
    if text[start:].strip():
        out.append(text[start:].strip())
    return out

@register_cell_magic
def sql(line, cell):
    """Run every statement in the cell; print rows for statements that return them."""
    expect_error = "expect_error" in line
    for stmt in _statements(cell):
        try:
            cur = con.execute(stmt)
            if cur.description:
                _table([d[0] for d in cur.description], cur.fetchall())
        except sqlite3.Error as exc:
            if not expect_error:
                raise
            print(f"ERROR: {exc}\n")

In [2]:
%%sql
CREATE TABLE web_users (email TEXT, name TEXT);
CREATE TABLE app_users (email TEXT, name TEXT);
INSERT INTO web_users VALUES
 ('ann@x.com','Ann'),('ben@x.com','Ben'),('cy@x.com','Cy'),('cy@x.com','Cy'),(NULL,'Ghost');
INSERT INTO app_users VALUES
 ('ben@x.com','Ben'),('cy@x.com','Cy'),('di@x.com','Di'),(NULL,'Ghost');
SELECT 'web' AS src, email, name FROM web_users
UNION ALL SELECT 'app', email, name FROM app_users
ORDER BY src, name, email;

src | email     | name 
----+-----------+------
app | ben@x.com | Ben  
app | cy@x.com  | Cy   
app | di@x.com  | Di   
app | NULL      | Ghost
web | ann@x.com | Ann  
web | ben@x.com | Ben  
web | cy@x.com  | Cy   
web | cy@x.com  | Cy   
web | NULL      | Ghost
(9 rows)



## 02. UNION ALL
- Keeps every row from both sides, including duplicates. Cheapest option.

In [3]:
%%sql
SELECT email FROM web_users
UNION ALL
SELECT email FROM app_users
ORDER BY email;

email    
---------
NULL     
NULL     
ann@x.com
ben@x.com
ben@x.com
cy@x.com 
cy@x.com 
cy@x.com 
di@x.com 
(9 rows)



## 03. UNION (removes duplicates)
- Same stacking, but duplicate rows collapse into one. Cy appears once, NULL appears once.
- Duplicates are removed across both sides AND within each side.

In [4]:
%%sql
SELECT email FROM web_users
UNION
SELECT email FROM app_users
ORDER BY email;

email    
---------
NULL     
ann@x.com
ben@x.com
cy@x.com 
di@x.com 
(5 rows)



## 04. INTERSECT
- Rows that appear on both sides, de-duplicated.
- Note the NULL row matches NULL here. In a join, `NULL = NULL` would not match.

In [5]:
%%sql
SELECT email FROM web_users
INTERSECT
SELECT email FROM app_users
ORDER BY email;

email    
---------
NULL     
ben@x.com
cy@x.com 
(3 rows)



## 05. EXCEPT
- Rows on the left that are not on the right, de-duplicated.
- Order matters: run it both ways to see the difference.

In [6]:
%%sql
SELECT 'web only' AS side, email FROM (SELECT email FROM web_users EXCEPT SELECT email FROM app_users)
UNION ALL
SELECT 'app only', email FROM (SELECT email FROM app_users EXCEPT SELECT email FROM web_users)
ORDER BY side, email;

side     | email    
---------+----------
app only | di@x.com 
web only | ann@x.com
(2 rows)



## 06. Duplicates side by side
- Count what each operator returns from the same inputs.

In [7]:
%%sql
SELECT 'UNION ALL' AS op, COUNT(*) AS rows_returned FROM (SELECT email FROM web_users UNION ALL SELECT email FROM app_users)
UNION ALL SELECT 'UNION',     COUNT(*) FROM (SELECT email FROM web_users UNION SELECT email FROM app_users)
UNION ALL SELECT 'INTERSECT', COUNT(*) FROM (SELECT email FROM web_users INTERSECT SELECT email FROM app_users)
UNION ALL SELECT 'EXCEPT',    COUNT(*) FROM (SELECT email FROM web_users EXCEPT SELECT email FROM app_users)
ORDER BY op;

op        | rows_returned
----------+--------------
EXCEPT    | 1            
INTERSECT | 3            
UNION     | 5            
UNION ALL | 9            
(4 rows)



## 07. ORDER BY and LIMIT with compound selects
- Only one ORDER BY / LIMIT, at the end, covering the whole result.
- ORDER BY can only use output column names (or numbers) of the compound result.
- To sort or limit one side first, wrap it in a subquery.

In [8]:
%%sql
SELECT name AS person FROM web_users
UNION
SELECT name FROM app_users
ORDER BY person DESC
LIMIT 3;
-- Limit one side only, using a subquery
SELECT name FROM (SELECT name FROM web_users ORDER BY name LIMIT 2)
UNION ALL
SELECT name FROM app_users
ORDER BY name;

person
------
Ghost 
Di    
Cy    
(3 rows)

name 
-----
Ann  
Ben  
Ben  
Cy   
Di   
Ghost
(6 rows)



## 08. Column alignment and names
- Columns pair up by position. Names come from the FIRST select. Counts must match.
- Bad vs good: swapped column order mixes names into the email column with no error.

In [9]:
%%sql expect_error
SELECT email, name FROM web_users UNION SELECT email FROM app_users;

ERROR: SELECTs to the left and right of UNION do not have the same number of result columns



In [10]:
%%sql
-- Bad: wrong order, no error, wrong meaning
SELECT email, name FROM web_users WHERE name = 'Ann'
UNION ALL
SELECT name, email FROM app_users WHERE name = 'Di'
ORDER BY 1;
-- Good: same order, explicit names
SELECT email, name FROM web_users WHERE name = 'Ann'
UNION ALL
SELECT email, name FROM app_users WHERE name = 'Di'
ORDER BY 1;

email     | name    
----------+---------
Di        | di@x.com
ann@x.com | Ann     
(2 rows)

email     | name
----------+-----
ann@x.com | Ann 
di@x.com  | Di  
(2 rows)



## 09. Type affinity in set operations
- SQLite does not check types. An integer 1 and the text '1' are different values, so `UNION` keeps both. They look the same when printed, so the first query shows `typeof`.
- Cast to a common type before combining.

In [11]:
%%sql
SELECT v, typeof(v) AS t FROM (SELECT 1 AS v UNION SELECT '1') ORDER BY t;
SELECT CAST(1 AS TEXT) AS v UNION SELECT '1';

v | t      
--+--------
1 | integer
1 | text   
(2 rows)

v
-
1
(1 row)



## 10. Precedence: left to right
- In SQLite all four operators have equal precedence and evaluate left to right. (Standard SQL says INTERSECT binds tighter. Do not rely on it.)
- To get a different grouping, put part of it in a subquery.

In [12]:
%%sql
CREATE TABLE a (v INTEGER);
CREATE TABLE b (v INTEGER);
CREATE TABLE c (v INTEGER);
INSERT INTO a VALUES (1),(2),(3);
INSERT INTO b VALUES (2),(3),(4);
INSERT INTO c VALUES (3),(4),(5);
-- left to right: (a UNION b) INTERSECT c
SELECT v FROM a UNION SELECT v FROM b INTERSECT SELECT v FROM c ORDER BY v;
-- grouped the other way: a UNION (b INTERSECT c)
SELECT v FROM a UNION SELECT v FROM (SELECT v FROM b INTERSECT SELECT v FROM c) ORDER BY v;

v
-
3
4
(2 rows)

v
-
1
2
3
4
(4 rows)



## 11. Set operations vs joins
- `INTERSECT` is like an inner join on all columns, but treats NULLs as equal and removes duplicates.
- `EXCEPT` is like an anti-join (`LEFT JOIN ... IS NULL`), with the same NULL difference. The join query below returns the NULL email, but `EXCEPT` would not, because NULL matches NULL there.
- Use set operations when comparing whole rows. Use joins when you need columns from both sides (see 02-Querying/01-Joins).

In [13]:
%%sql
SELECT DISTINCT w.email FROM web_users w JOIN app_users a ON a.email = w.email ORDER BY w.email;
SELECT DISTINCT w.email FROM web_users w LEFT JOIN app_users a ON a.email = w.email WHERE a.email IS NULL ORDER BY w.email;

email    
---------
ben@x.com
cy@x.com 
(2 rows)

email    
---------
NULL     
ann@x.com
(2 rows)



## 12. Finding differences between two tables
- A reconciliation pattern: rows in one table but not the other, in both directions, with a label.
- Compares whole rows, so a changed price shows up as one "old only" and one "new only".

In [14]:
%%sql
CREATE TABLE prices_old (sku TEXT, price REAL);
CREATE TABLE prices_new (sku TEXT, price REAL);
INSERT INTO prices_old VALUES ('A',10),('B',20),('C',30);
INSERT INTO prices_new VALUES ('A',10),('B',25),('D',40);
SELECT 'old only' AS diff, sku, price FROM (SELECT * FROM prices_old EXCEPT SELECT * FROM prices_new)
UNION ALL
SELECT 'new only', sku, price FROM (SELECT * FROM prices_new EXCEPT SELECT * FROM prices_old)
ORDER BY sku, diff;

diff     | sku | price
---------+-----+------
new only | B   | 25.0 
old only | B   | 20.0 
old only | C   | 30.0 
new only | D   | 40.0 
(4 rows)



## 13. NULL handling
- In `UNION`, `INTERSECT`, `EXCEPT`, `DISTINCT`, NULL equals NULL.
- In `=`, joins and `IN`, NULL equals nothing.
- That is why `INTERSECT` finds the NULL email but a plain join does not.

In [15]:
%%sql
SELECT COUNT(*) AS intersect_has_null FROM (SELECT email FROM web_users INTERSECT SELECT email FROM app_users) WHERE email IS NULL;
SELECT COUNT(*) AS join_matches_null FROM web_users w JOIN app_users a ON a.email = w.email WHERE w.email IS NULL;
-- null-safe join comparison, for reference
SELECT COUNT(*) AS join_with_IS FROM web_users w JOIN app_users a ON a.email IS w.email WHERE w.email IS NULL;

intersect_has_null
------------------
1                 
(1 row)

join_matches_null
-----------------
0                
(1 row)

join_with_IS
------------
1           
(1 row)



## 14. Emulating FULL OUTER JOIN
- Recent SQLite (3.39+) has `FULL OUTER JOIN`, but many databases and older SQLite do not, and the pattern is worth knowing.
- Recipe: `LEFT JOIN`, then `UNION ALL` the right-only rows (`WHERE left IS NULL`).

In [16]:
%%sql
SELECT o.sku AS sku, o.price AS old_price, n.price AS new_price
FROM prices_old o LEFT JOIN prices_new n ON n.sku = o.sku
UNION ALL
SELECT n.sku, NULL, n.price
FROM prices_new n LEFT JOIN prices_old o ON o.sku = n.sku
WHERE o.sku IS NULL
ORDER BY sku;

sku | old_price | new_price
----+-----------+----------
A   | 10.0      | 10.0     
B   | 20.0      | 25.0     
C   | 30.0      | NULL     
D   | NULL      | 40.0     
(4 rows)



## 15. Emulating INTERSECT ALL (keep duplicate counts)
- `INTERSECT` collapses duplicates. To keep the smaller count of each value, number the duplicates and intersect the numbered rows.
- `EXCEPT ALL` works the same way with `EXCEPT`. Window functions are covered in 03-Window-Functions/01-Window-Functions.

In [17]:
%%sql
CREATE TABLE bag1 (x TEXT);
CREATE TABLE bag2 (x TEXT);
INSERT INTO bag1 VALUES ('a'),('a'),('a'),('b');
INSERT INTO bag2 VALUES ('a'),('a'),('b'),('b'),('c');
SELECT x, COUNT(*) AS copies FROM (
    SELECT x, ROW_NUMBER() OVER (PARTITION BY x ORDER BY x) AS n FROM bag1
    INTERSECT
    SELECT x, ROW_NUMBER() OVER (PARTITION BY x ORDER BY x) AS n FROM bag2
)
GROUP BY x ORDER BY x;

x | copies
--+-------
a | 2     
b | 1     
(2 rows)



## 16. Practical: merging partitioned tables
- Monthly tables with the same shape combine with `UNION ALL`. Add a literal column to remember where each row came from.

In [18]:
%%sql
CREATE TABLE sales_jan (item TEXT, qty INTEGER);
CREATE TABLE sales_feb (item TEXT, qty INTEGER);
INSERT INTO sales_jan VALUES ('pen',5),('ink',2);
INSERT INTO sales_feb VALUES ('pen',7),('pad',1);
SELECT item, SUM(qty) AS total_qty, COUNT(*) AS months_seen FROM (
    SELECT 'jan' AS month, item, qty FROM sales_jan
    UNION ALL
    SELECT 'feb', item, qty FROM sales_feb
)
GROUP BY item ORDER BY item;

item | total_qty | months_seen
-----+-----------+------------
ink  | 2         | 1          
pad  | 1         | 1          
pen  | 12        | 2          
(3 rows)



## 17. Quick chooser
- Stack everything: `UNION ALL`. Stack and de-duplicate: `UNION`.
- In both lists: `INTERSECT`. In one list only: `EXCEPT`.
- Need other columns too: use a join or `EXISTS` instead (see 02-Querying/03-Subqueries-And-CTEs).